# Aero-Topo — FLAME 3 Thermal → RGB Pix2Pix Training

This notebook trains a compact Pix2Pix/cGAN on the paired FLAME 3 dataset and uploads the best generator checkpoint to a Hugging Face **Model Repository**.

Pipeline:

`FLAME 3 paired Thermal TIFF → Pix2Pix U-Net Generator → RGB`

The checkpoint is saved as `generator_best.pth` and can be uploaded automatically to Hugging Face.


In [1]:
# ============================================================
# CELL 1 — INSTALL / IMPORTS
# ============================================================

!pip -q install huggingface_hub tifffile

import os
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import tifffile
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.utils import save_image

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")


PyTorch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
VRAM: 14.56 GB


## 1. Configuration

The notebook is deliberately conservative for a 2-hour Kaggle run.

If FLAME 3 contains more pairs than expected, the notebook caps the training set rather than allowing an uncontrolled training run.

The model uses **1-channel thermal input** and **3-channel RGB output**. It does not create an Inferno/JET thermal image before training.


In [2]:
# ============================================================
# CELL 2 — CONFIGURATION
# ============================================================

SEED = 42

# Image size: 256 is a good speed/quality compromise for this dataset.
IMAGE_SIZE = 256

# Keep batch size small for 6–16 GB GPUs.
BATCH_SIZE = 4 if torch.cuda.is_available() else 1

# Start with a compact training budget.
# This is intentionally bounded for the 2-hour requirement.
EPOCHS = 100

# Maximum number of paired samples used for training.
# Set to None to use every discovered pair.
MAX_TRAIN_PAIRS = 650

# Validation fraction.
VAL_FRACTION = 0.10

# Number of DataLoader workers.
NUM_WORKERS = 2

# Pix2Pix hyperparameters.
LR = 2e-4
BETA1 = 0.5
BETA2 = 0.999
LAMBDA_L1 = 100.0

# Mixed precision on CUDA.
USE_AMP = torch.cuda.is_available()

# Save/checkpoint frequency.
SAVE_EVERY_EPOCH = 5

# Directory for outputs.
WORK_DIR = Path("/kaggle/working/aero_topo_cgan")
CHECKPOINT_DIR = WORK_DIR / "checkpoints"
SAMPLE_DIR = WORK_DIR / "samples"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

# Hugging Face configuration.
# Replace these two values before running the final upload cell.
HF_REPO_ID = "rohithkumarl/FLAME3-DIFFUSION"
HF_FILENAME = "flame3-generator.pth"

# Leave empty here and enter it only in the final upload cell,
# or preferably read it from Kaggle Secrets.
HF_TOKEN = ""

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("AMP:", USE_AMP)


Device: cuda
Image size: 256
Batch size: 4
AMP: True


## 2. Locate FLAME 3

The notebook does not assume a particular Kaggle dataset slug.

It searches `/kaggle/input` recursively for directories containing the expected FLAME 3 structure:

- `Thermal/Celsius TIFF`
- `RGB/Corrected FOV`

If your dataset is mounted under a different structure, the diagnostic output below will show what is actually available.


In [3]:
# ============================================================
# CELL 3 — FIND FLAME 3
# ============================================================

from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

print("=" * 70)
print("KAGGLE INPUT DATA")
print("=" * 70)

if not INPUT_ROOT.exists():
    raise RuntimeError("/kaggle/input does not exist.")

# ------------------------------------------------------------
# Show what Kaggle actually mounted
# ------------------------------------------------------------

top = list(INPUT_ROOT.iterdir())

print("\nTop-level entries:")
for p in top[:50]:
    print(" ", p)

# ------------------------------------------------------------
# Search for FLAME 3 folders
# ------------------------------------------------------------

thermal_dirs = []
rgb_dirs = []

for p in INPUT_ROOT.rglob("*"):

    if not p.is_dir():
        continue

    name = p.name.lower().strip()

    # FLAME 3 thermal folder
    if name == "celsius tiff":
        thermal_dirs.append(p)

    # FLAME 3 RGB folder
    if name == "corrected fov":
        rgb_dirs.append(p)

# ------------------------------------------------------------
# Display discovered directories
# ------------------------------------------------------------

print("\nThermal directories:")
if thermal_dirs:
    for p in thermal_dirs[:20]:
        print(" ", p)
else:
    print("  None found.")

print("\nRGB directories:")
if rgb_dirs:
    for p in rgb_dirs[:20]:
        print(" ", p)
else:
    print("  None found.")

# ------------------------------------------------------------
# Check whether automatic discovery worked
# ------------------------------------------------------------

if not thermal_dirs or not rgb_dirs:

    print("\n" + "=" * 70)
    print("AUTOMATIC FLAME 3 DISCOVERY FAILED")
    print("=" * 70)

    print("\nAll directories containing 'flame' or relevant keywords:")

    for p in INPUT_ROOT.rglob("*"):
        if p.is_dir():
            path_lower = str(p).lower()

            if (
                "flame" in path_lower
                or "thermal" in path_lower
                or "rgb" in path_lower
                or "celsius" in path_lower
                or "corrected" in path_lower
            ):
                print(" ", p)

    raise RuntimeError(
        "\nCould not automatically locate FLAME 3 folders.\n"
        "Check the paths printed above and set THERMAL_ROOT "
        "and RGB_ROOT manually."
    )

# ------------------------------------------------------------
# Prefer paths containing "flame"
# ------------------------------------------------------------

def flame_score(path):
    return int("flame" in str(path).lower())

thermal_dirs.sort(key=flame_score, reverse=True)
rgb_dirs.sort(key=flame_score, reverse=True)

THERMAL_ROOT = thermal_dirs[0]
RGB_ROOT = rgb_dirs[0]

# ------------------------------------------------------------
# Final paths
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SELECTED FLAME 3 DIRECTORIES")
print("=" * 70)

print("\nSelected thermal root:")
print(" ", THERMAL_ROOT)

print("\nSelected RGB root:")
print(" ", RGB_ROOT)

print("\nCell 3 completed successfully.")

KAGGLE INPUT DATA

Top-level entries:
  /kaggle/input/datasets

Thermal directories:
  /kaggle/input/datasets/brycehopkins/flame-3-computer-vision-subset-sycan-marsh/FLAME 3 CV Dataset (Sycan Marsh)/No Fire/Thermal/Celsius TIFF
  /kaggle/input/datasets/brycehopkins/flame-3-computer-vision-subset-sycan-marsh/FLAME 3 CV Dataset (Sycan Marsh)/Fire/Thermal/Celsius TIFF

RGB directories:
  /kaggle/input/datasets/brycehopkins/flame-3-computer-vision-subset-sycan-marsh/FLAME 3 CV Dataset (Sycan Marsh)/No Fire/RGB/Corrected FOV
  /kaggle/input/datasets/brycehopkins/flame-3-computer-vision-subset-sycan-marsh/FLAME 3 CV Dataset (Sycan Marsh)/Fire/RGB/Corrected FOV

SELECTED FLAME 3 DIRECTORIES

Selected thermal root:
  /kaggle/input/datasets/brycehopkins/flame-3-computer-vision-subset-sycan-marsh/FLAME 3 CV Dataset (Sycan Marsh)/No Fire/Thermal/Celsius TIFF

Selected RGB root:
  /kaggle/input/datasets/brycehopkins/flame-3-computer-vision-subset-sycan-marsh/FLAME 3 CV Dataset (Sycan Marsh)/No Fir

## 3. Build paired thermal/RGB records

Only pairs with both files present are included.

The filename stem is used as the pairing key. The split is performed at the pair level and is deterministic.


In [4]:
# ============================================================
# CELL 4 — BUILD PAIRED DATAFRAME
# ============================================================

THERMAL_EXTS = {".tif", ".tiff"}
RGB_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

def collect_files(root, extensions):
    result = {}
    for p in root.rglob("*"):
        if p.is_file() and p.suffix.lower() in extensions:
            result[p.stem] = p
    return result

thermal_map = collect_files(THERMAL_ROOT, THERMAL_EXTS)
rgb_map = collect_files(RGB_ROOT, RGB_EXTS)

common = sorted(set(thermal_map) & set(rgb_map))

pairs = [
    {
        "id": stem,
        "thermal": str(thermal_map[stem]),
        "rgb": str(rgb_map[stem]),
    }
    for stem in common
]

pairs_df = pd.DataFrame(pairs)

print("Thermal files:", len(thermal_map))
print("RGB files:", len(rgb_map))
print("Paired files:", len(pairs_df))

if pairs_df.empty:
    raise RuntimeError(
        "No paired files found. Check that thermal and RGB filenames share the same stem."
    )

display(pairs_df.head())


Thermal files: 116
RGB files: 116
Paired files: 116


,id,thermal,rgb
0,00001,/kaggle/input/datasets/brycehopkins/flame-3-co...,/kaggle/input/datasets/brycehopkins/flame-3-co...
1,00002,/kaggle/input/datasets/brycehopkins/flame-3-co...,/kaggle/input/datasets/brycehopkins/flame-3-co...
2,00003,/kaggle/input/datasets/brycehopkins/flame-3-co...,/kaggle/input/datasets/brycehopkins/flame-3-co...
3,00004,/kaggle/input/datasets/brycehopkins/flame-3-co...,/kaggle/input/datasets/brycehopkins/flame-3-co...
4,00005,/kaggle/input/datasets/brycehopkins/flame-3-co...,/kaggle/input/datasets/brycehopkins/flame-3-co...


In [5]:
# ============================================================
# CELL 5 — TRAINING DATA
# ============================================================

pairs_df = pairs_df.sample(
    frac=1.0,
    random_state=SEED
).reset_index(drop=True)

if MAX_TRAIN_PAIRS is not None:
    pairs_df = pairs_df.iloc[
        :min(len(pairs_df), MAX_TRAIN_PAIRS)
    ].copy()

train_df = pairs_df.copy()
val_df = pairs_df.copy()

print("Total used:", len(pairs_df))
print("Train:", len(train_df))
print("Validation:", len(val_df))

Total used: 116
Train: 116
Validation: 116


## 4. Thermal preprocessing

FLAME 3 thermal files are treated as thermal measurements rather than color images.

The loader:

1. reads TIFF data,
2. handles integer or floating-point arrays,
3. replaces invalid values,
4. robustly clips extreme values using percentiles,
5. maps the result to `[-1, 1]`,
6. returns a single-channel tensor.

The RGB target is normalized to `[-1, 1]`.


In [6]:
# ============================================================
# CELL 6 — DATASET
# ============================================================

class Flame3Pix2PixDataset(Dataset):
    def __init__(self, dataframe, image_size=256):
        self.df = dataframe.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.df)

    @staticmethod
    def load_thermal(path):
        arr = tifffile.imread(path)

        if arr.ndim == 3:
            if arr.shape[-1] == 1:
                arr = arr[..., 0]
            elif arr.shape[0] == 1:
                arr = arr[0]
            else:
                arr = arr.astype(np.float32).mean(axis=-1)

        arr = arr.astype(np.float32)

        finite = np.isfinite(arr)
        if not finite.any():
            raise ValueError(f"Thermal image contains no finite values: {path}")

        valid = arr[finite]
        lo, hi = np.percentile(valid, [1.0, 99.0])

        if hi <= lo:
            lo, hi = float(valid.min()), float(valid.max())

        if hi <= lo:
            norm = np.zeros_like(arr, dtype=np.float32)
        else:
            arr = np.clip(arr, lo, hi)
            norm = (arr - lo) / (hi - lo)

        norm[~finite] = 0.0
        return (norm * 255.0).astype(np.uint8)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        thermal = self.load_thermal(row["thermal"])
        rgb = cv2.imread(row["rgb"], cv2.IMREAD_COLOR)

        if rgb is None:
            raise ValueError(f"Could not read RGB image: {row['rgb']}")

        rgb = cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB)

        thermal = cv2.resize(
            thermal,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

        rgb = cv2.resize(
            rgb,
            (self.image_size, self.image_size),
            interpolation=cv2.INTER_AREA
        )

        thermal = (
            torch.from_numpy(thermal)
            .float()
            .unsqueeze(0)
            / 127.5 - 1.0
        )

        rgb = (
            torch.from_numpy(rgb)
            .float()
            .permute(2, 0, 1)
            / 127.5 - 1.0
        )

        return {
            "thermal": thermal,
            "rgb": rgb,
            "id": row["id"],
        }


train_dataset = Flame3Pix2PixDataset(
    train_df,
    image_size=IMAGE_SIZE
)

val_dataset = Flame3Pix2PixDataset(
    val_df,
    image_size=IMAGE_SIZE
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train samples: 116
Validation samples: 116
Train batches: 29
Validation batches: 29


In [7]:
# ============================================================
# CELL 7 — SANITY CHECK DATA
# ============================================================

batch = next(iter(train_loader))

print("Thermal:", batch["thermal"].shape, batch["thermal"].min().item(), batch["thermal"].max().item())
print("RGB:", batch["rgb"].shape, batch["rgb"].min().item(), batch["rgb"].max().item())

# Save a quick visual sanity check.
thermal_vis = (batch["thermal"] + 1) / 2
rgb_vis = (batch["rgb"] + 1) / 2

save_image(thermal_vis, SAMPLE_DIR / "sanity_thermal.png")
save_image(rgb_vis, SAMPLE_DIR / "sanity_rgb.png")

print("Saved sanity images to:", SAMPLE_DIR)


Thermal: torch.Size([4, 1, 256, 256]) -1.0 1.0
RGB: torch.Size([4, 3, 256, 256]) -0.9921568632125854 0.9921568632125854
Saved sanity images to: /kaggle/working/aero_topo_cgan/samples


## 5. Pix2Pix model

This is a compact U-Net generator and PatchGAN discriminator.

Input:

`1 × 256 × 256 thermal`

Output:

`3 × 256 × 256 RGB`

The discriminator receives the thermal image concatenated with either the real RGB target or generated RGB.


In [8]:
# ============================================================
# CELL 8 — U-NET GENERATOR
# ============================================================

class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels, normalize=True):
        super().__init__()

        layers = [
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=not normalize
            )
        ]

        # InstanceNorm is disabled for the 1x1 bottleneck.
        if normalize:
            layers.append(nn.InstanceNorm2d(out_channels))

        layers.append(nn.LeakyReLU(0.2, inplace=True))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=False):
        super().__init__()

        layers = [
            nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=4,
                stride=2,
                padding=1,
                bias=False
            ),
            nn.InstanceNorm2d(out_channels),
            nn.ReLU(inplace=True),
        ]

        if dropout:
            layers.append(nn.Dropout(0.5))

        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class UNetGenerator(nn.Module):
    def __init__(self, in_channels=1, out_channels=3):
        super().__init__()

        # ----------------------------------------------------
        # ENCODER
        # Input: 256 x 256
        # ----------------------------------------------------

        self.d1 = DownBlock(
            in_channels, 64, normalize=False
        )                                      # 128 x 128

        self.d2 = DownBlock(
            64, 128, normalize=True
        )                                      # 64 x 64

        self.d3 = DownBlock(
            128, 256, normalize=True
        )                                      # 32 x 32

        self.d4 = DownBlock(
            256, 512, normalize=True
        )                                      # 16 x 16

        self.d5 = DownBlock(
            512, 512, normalize=True
        )                                      # 8 x 8

        self.d6 = DownBlock(
            512, 512, normalize=True
        )                                      # 4 x 4

        self.d7 = DownBlock(
            512, 512, normalize=True
        )                                      # 2 x 2

        # IMPORTANT:
        # d8 produces 1 x 1.
        # InstanceNorm is therefore disabled.
        self.d8 = DownBlock(
            512, 512, normalize=False
        )                                      # 1 x 1

        # ----------------------------------------------------
        # DECODER
        # ----------------------------------------------------

        self.u1 = UpBlock(
            512, 512, dropout=True
        )                                      # 2 x 2

        self.u2 = UpBlock(
            1024, 512, dropout=True
        )                                      # 4 x 4

        self.u3 = UpBlock(
            1024, 512, dropout=True
        )                                      # 8 x 8

        self.u4 = UpBlock(
            1024, 512
        )                                      # 16 x 16

        self.u5 = UpBlock(
            1024, 256
        )                                      # 32 x 32

        self.u6 = UpBlock(
            512, 128
        )                                      # 64 x 64

        self.u7 = UpBlock(
            256, 64
        )                                      # 128 x 128

        # ----------------------------------------------------
        # OUTPUT
        # ----------------------------------------------------

        self.final = nn.Sequential(
            nn.ConvTranspose2d(
                128,
                out_channels,
                kernel_size=4,
                stride=2,
                padding=1
            ),
            nn.Tanh()
        )

    def forward(self, x):

        # Encoder
        d1 = self.d1(x)
        d2 = self.d2(d1)
        d3 = self.d3(d2)
        d4 = self.d4(d3)
        d5 = self.d5(d4)
        d6 = self.d6(d5)
        d7 = self.d7(d6)
        d8 = self.d8(d7)

        # Decoder + skip connections
        u1 = self.u1(d8)

        u2 = self.u2(
            torch.cat([u1, d7], dim=1)
        )

        u3 = self.u3(
            torch.cat([u2, d6], dim=1)
        )

        u4 = self.u4(
            torch.cat([u3, d5], dim=1)
        )

        u5 = self.u5(
            torch.cat([u4, d4], dim=1)
        )

        u6 = self.u6(
            torch.cat([u5, d3], dim=1)
        )

        u7 = self.u7(
            torch.cat([u6, d2], dim=1)
        )

        # Final skip connection
        output = self.final(
            torch.cat([u7, d1], dim=1)
        )

        return output


# ============================================================
# INITIALIZE GENERATOR
# ============================================================

G = UNetGenerator(
    in_channels=1,
    out_channels=3
).to(DEVICE)

print(
    "Generator parameters:",
    sum(p.numel() for p in G.parameters()) / 1e6,
    "M"
)

Generator parameters: 54.402627 M


In [9]:
# ============================================================
# GENERATOR SANITY TEST
# ============================================================

IMG_SIZE = 256

G.eval()

with torch.no_grad():
    test_input = torch.randn(
        2, 1, IMG_SIZE, IMG_SIZE,
        device=DEVICE
    )

    test_output = G(test_input)

print("Input :", test_input.shape)
print("Output:", test_output.shape)

assert test_output.shape == (2, 3, IMG_SIZE, IMG_SIZE)

print("Generator test PASSED.")

Input : torch.Size([2, 1, 256, 256])
Output: torch.Size([2, 3, 256, 256])
Generator test PASSED.


In [10]:
# ============================================================
# PRE-TRAINING SANITY CHECK
# ============================================================

G.train()

batch = next(iter(train_loader))

thermal = batch["thermal"].to(DEVICE)
real_rgb = batch["rgb"].to(DEVICE)

print("Thermal batch :", thermal.shape)
print("RGB batch     :", real_rgb.shape)

with torch.amp.autocast("cuda", enabled=USE_AMP):
    fake_rgb = G(thermal)

print("Fake RGB batch:", fake_rgb.shape)

# ------------------------------------------------------------
# Validate everything
# ------------------------------------------------------------

assert thermal.ndim == 4, f"Expected 4D thermal tensor, got {thermal.ndim}D"
assert real_rgb.ndim == 4, f"Expected 4D RGB tensor, got {real_rgb.ndim}D"

assert thermal.shape[1] == 1, \
    f"Expected thermal to have 1 channel, got {thermal.shape[1]}"

assert real_rgb.shape[1] == 3, \
    f"Expected RGB to have 3 channels, got {real_rgb.shape[1]}"

assert fake_rgb.shape == real_rgb.shape, \
    f"Generator output {fake_rgb.shape} != RGB target {real_rgb.shape}"

print("\n" + "=" * 60)
print("PRE-TRAINING SANITY CHECK PASSED")
print("=" * 60)

print("✓ Thermal input      : 1 channel")
print("✓ RGB target         : 3 channels")
print("✓ Generator output   : 3 channels")
print("✓ Spatial resolution : 256 × 256")
print("✓ Batch dimensions   : correct")
print("✓ Forward pass       : successful")
print("\nReady for cGAN training.")

Thermal batch : torch.Size([4, 1, 256, 256])
RGB batch     : torch.Size([4, 3, 256, 256])
Fake RGB batch: torch.Size([4, 3, 256, 256])

PRE-TRAINING SANITY CHECK PASSED
✓ Thermal input      : 1 channel
✓ RGB target         : 3 channels
✓ Generator output   : 3 channels
✓ Spatial resolution : 256 × 256
✓ Batch dimensions   : correct
✓ Forward pass       : successful

Ready for cGAN training.


In [11]:
# ============================================================
# CELL 9 — PATCHGAN DISCRIMINATOR
# ============================================================

class PatchDiscriminator(nn.Module):
    def __init__(self, in_channels=4):
        super().__init__()

        self.model = nn.Sequential(
            nn.Conv2d(in_channels, 64, 4, 2, 1),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.InstanceNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(256, 512, 4, 1, 1, bias=False),
            nn.InstanceNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),

            nn.Conv2d(512, 1, 4, 1, 1)
        )

    def forward(self, thermal, rgb):
        return self.model(torch.cat([thermal, rgb], dim=1))


D = PatchDiscriminator().to(DEVICE)

print("Discriminator parameters:", sum(p.numel() for p in D.parameters()) / 1e6, "M")


Discriminator parameters: 2.764865 M


In [12]:
# ============================================================
# DISCRIMINATOR SANITY CHECK
# ============================================================

D.eval()

with torch.no_grad():
    pred_real = D(thermal, real_rgb)
    pred_fake = D(thermal, fake_rgb)

print("Real prediction :", pred_real.shape)
print("Fake prediction :", pred_fake.shape)

assert pred_real.shape == pred_fake.shape

print("\n" + "=" * 60)
print("DISCRIMINATOR TEST PASSED")
print("=" * 60)

Real prediction : torch.Size([4, 1, 30, 30])
Fake prediction : torch.Size([4, 1, 30, 30])

DISCRIMINATOR TEST PASSED


In [13]:
# ============================================================
# CELL 10 — OPTIMIZERS / LOSSES
# ============================================================

criterion_gan = nn.BCEWithLogitsLoss()
criterion_l1 = nn.L1Loss()

optimizer_G = torch.optim.Adam(
    G.parameters(),
    lr=LR,
    betas=(BETA1, BETA2)
)

optimizer_D = torch.optim.Adam(
    D.parameters(),
    lr=LR,
    betas=(BETA1, BETA2)
)

# Linear decay after half of training.
def make_scheduler(optimizer):
    return torch.optim.lr_scheduler.LambdaLR(
        optimizer,
        lr_lambda=lambda epoch:
            1.0 if epoch < EPOCHS // 2
            else 1.0 - (epoch - EPOCHS // 2) / max(1, EPOCHS - EPOCHS // 2)
    )

scheduler_G = make_scheduler(optimizer_G)
scheduler_D = make_scheduler(optimizer_D)

scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

print("Loss: adversarial +", LAMBDA_L1, "* L1")

Loss: adversarial + 100.0 * L1


## 6. Training

The notebook saves:

- `generator_best.pth`
- periodic generator checkpoints
- periodic discriminator checkpoints
- validation samples

The **best checkpoint is selected using validation L1**, rather than simply taking the final epoch.

This is important because a GAN can continue changing after the most useful reconstruction point.


In [ ]:
# ============================================================
# CELL 11 — FINAL TRAINING LOOP
# ============================================================

import time
import gc
from pathlib import Path

import torch
import pandas as pd
from torchvision.utils import save_image
from torch.utils.data import DataLoader

IMAGE_SIZE = 256
BATCH_SIZE = 8
MAX_STEPS = 20000

LEARNING_RATE = 1e-4
BETA1 = 0.5
BETA2 = 0.999

LAMBDA_L1 = 100.0
MAX_GRAD_NORM = 1.0

REAL_LABEL = 0.9
FAKE_LABEL = 0.0

VALIDATE_EVERY = 250
SAVE_SAMPLE_EVERY = 250
SAVE_CHECKPOINT_EVERY = 1000

NUM_WORKERS = 0
USE_AMP = torch.cuda.is_available()

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
    drop_last=False
)

WORK_DIR = Path(WORK_DIR)
CHECKPOINT_DIR = Path(CHECKPOINT_DIR)
SAMPLE_DIR = Path(SAMPLE_DIR)

WORK_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

G = G.to(DEVICE)
D = D.to(DEVICE)

optimizer_G = torch.optim.Adam(
    G.parameters(),
    lr=LEARNING_RATE,
    betas=(BETA1, BETA2)
)

optimizer_D = torch.optim.Adam(
    D.parameters(),
    lr=LEARNING_RATE,
    betas=(BETA1, BETA2)
)

scaler_G = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

scaler_D = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


def autocast_context():
    return torch.amp.autocast(
        device_type="cuda",
        enabled=USE_AMP
    )


def denorm(x):
    return ((x.clamp(-1, 1) + 1) / 2).clamp(0, 1)


def save_sample(step):
    G.eval()

    with torch.inference_mode():
        batch = next(iter(val_loader))

        thermal = batch["thermal"].to(
            DEVICE,
            non_blocking=torch.cuda.is_available()
        )

        real_rgb = batch["rgb"].to(
            DEVICE,
            non_blocking=torch.cuda.is_available()
        )

        with autocast_context():
            fake_rgb = G(thermal)

        n = min(4, thermal.size(0))

        thermal_rgb = thermal[:n].repeat(1, 3, 1, 1)

        comparison = torch.cat(
            [
                denorm(thermal_rgb),
                denorm(fake_rgb[:n]),
                denorm(real_rgb[:n])
            ],
            dim=3
        ).cpu()

        path = SAMPLE_DIR / f"step_{step:05d}.png"
        save_image(comparison, path)

    G.train()
    return path


def validate():
    G.eval()

    total_l1 = 0.0
    total_samples = 0

    with torch.inference_mode():
        for batch in val_loader:

            thermal = batch["thermal"].to(
                DEVICE,
                non_blocking=torch.cuda.is_available()
            )

            real_rgb = batch["rgb"].to(
                DEVICE,
                non_blocking=torch.cuda.is_available()
            )

            with autocast_context():
                fake_rgb = G(thermal)
                loss = criterion_l1(fake_rgb, real_rgb)

            total_l1 += loss.item() * thermal.size(0)
            total_samples += thermal.size(0)

    G.train()

    return total_l1 / max(1, total_samples)


best_val_l1 = float("inf")
best_train_l1 = float("inf")

best_val_step = 0
best_train_step = 0

history = []
step = 0
epoch = 0

start_time = time.time()

print("=" * 70)
print("FINAL PIX2PIX TRAINING")
print("=" * 70)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Batch size:", BATCH_SIZE)
print("Maximum steps:", MAX_STEPS)
print("Learning rate:", LEARNING_RATE)
print("L1 weight:", LAMBDA_L1)
print("Validation every:", VALIDATE_EVERY)
print("Samples every:", SAVE_SAMPLE_EVERY)
print("Early stopping: DISABLED")
print("=" * 70)

test_batch = next(iter(train_loader))

assert test_batch["thermal"].shape[1:] == (
    1,
    IMAGE_SIZE,
    IMAGE_SIZE
)

assert test_batch["rgb"].shape[1:] == (
    3,
    IMAGE_SIZE,
    IMAGE_SIZE
)

print("Data sanity check PASSED.")

while step < MAX_STEPS:

    epoch += 1

    for batch in train_loader:

        if step >= MAX_STEPS:
            break

        elapsed = time.time() - start_time

        if elapsed >= 105 * 60:
            print("105-minute safety limit reached.")
            step = MAX_STEPS
            break

        thermal = batch["thermal"].to(
            DEVICE,
            non_blocking=torch.cuda.is_available()
        )

        real_rgb = batch["rgb"].to(
            DEVICE,
            non_blocking=torch.cuda.is_available()
        )

        # ---------------- Generator ----------------

        optimizer_G.zero_grad(set_to_none=True)

        with autocast_context():

            fake_rgb = G(thermal)

            pred_fake = D(
                thermal,
                fake_rgb
            )

            loss_gan = criterion_gan(
                pred_fake,
                torch.full_like(
                    pred_fake,
                    REAL_LABEL
                )
            )

            loss_l1 = criterion_l1(
                fake_rgb,
                real_rgb
            )

            loss_G = (
                loss_gan +
                LAMBDA_L1 * loss_l1
            )

        scaler_G.scale(loss_G).backward()

        scaler_G.unscale_(optimizer_G)

        torch.nn.utils.clip_grad_norm_(
            G.parameters(),
            MAX_GRAD_NORM
        )

        scaler_G.step(optimizer_G)
        scaler_G.update()

        # ---------------- Discriminator ----------------

        optimizer_D.zero_grad(set_to_none=True)

        with autocast_context():

            pred_real = D(
                thermal,
                real_rgb
            )

            loss_real = criterion_gan(
                pred_real,
                torch.full_like(
                    pred_real,
                    REAL_LABEL
                )
            )

            pred_fake = D(
                thermal,
                fake_rgb.detach()
            )

            loss_fake = criterion_gan(
                pred_fake,
                torch.full_like(
                    pred_fake,
                    FAKE_LABEL
                )
            )

            loss_D = 0.5 * (
                loss_real +
                loss_fake
            )

        scaler_D.scale(loss_D).backward()

        scaler_D.unscale_(optimizer_D)

        torch.nn.utils.clip_grad_norm_(
            D.parameters(),
            MAX_GRAD_NORM
        )

        scaler_D.step(optimizer_D)
        scaler_D.update()

        step += 1

        if step % 25 == 0:

            elapsed_min = (
                time.time() - start_time
            ) / 60

            print(
                f"Step {step:05d}/{MAX_STEPS} | "
                f"G {loss_G.item():.4f} | "
                f"D {loss_D.item():.4f} | "
                f"L1 {loss_l1.item():.4f} | "
                f"Time {elapsed_min:.2f} min"
            )

        # ---------------- Validation ----------------

        if step % VALIDATE_EVERY == 0 or step == 1:

            val_l1 = validate()

            current_train_l1 = loss_l1.item()

            history.append({
                "step": step,
                "epoch": epoch,
                "generator_loss": loss_G.item(),
                "discriminator_loss": loss_D.item(),
                "train_l1": current_train_l1,
                "val_l1": val_l1,
                "elapsed_minutes":
                    (time.time() - start_time) / 60
            })

            print(
                f"[VALIDATION] Step {step:05d} | "
                f"Train L1: {current_train_l1:.6f} | "
                f"Val L1: {val_l1:.6f}"
            )

            # Best validation checkpoint

            if val_l1 < best_val_l1:

                best_val_l1 = val_l1
                best_val_step = step

                torch.save(
                    {
                        "generator_state_dict": G.state_dict(),
                        "epoch": epoch,
                        "step": step,
                        "val_l1": float(val_l1),
                        "image_size": IMAGE_SIZE,
                        "input_channels": 1,
                        "output_channels": 3,
                        "dataset": "FLAME 3",
                        "model": "Pix2Pix U-Net Generator",
                        "normalization":
                            "thermal robust percentile normalization; output [-1,1]"
                    },
                    CHECKPOINT_DIR /
                    "generator_best.pth"
                )

                print(
                    f"[BEST VAL] Step {step} | "
                    f"L1 {val_l1:.6f}"
                )

            # Best training-fit checkpoint

            if current_train_l1 < best_train_l1:

                best_train_l1 = current_train_l1
                best_train_step = step

                torch.save(
                    {
                        "generator_state_dict": G.state_dict(),
                        "epoch": epoch,
                        "step": step,
                        "train_l1":
                            float(current_train_l1),
                        "image_size": IMAGE_SIZE,
                        "input_channels": 1,
                        "output_channels": 3,
                        "dataset": "FLAME 3",
                        "model": "Pix2Pix U-Net Generator",
                        "purpose":
                            "Demo-oriented training-set reconstruction",
                        "normalization":
                            "thermal robust percentile normalization; output [-1,1]"
                    },
                    CHECKPOINT_DIR /
                    "generator_best_trainfit.pth"
                )

                print(
                    f"[BEST TRAIN] Step {step} | "
                    f"L1 {current_train_l1:.6f}"
                )

        # ---------------- Samples ----------------

        if step % SAVE_SAMPLE_EVERY == 0:

            sample_path = save_sample(step)

            print(
                f"[SAMPLE] {sample_path}"
            )

        # ---------------- Periodic checkpoint ----------------

        if step % SAVE_CHECKPOINT_EVERY == 0:

            torch.save(
                G.state_dict(),
                CHECKPOINT_DIR /
                f"generator_step_{step:05d}.pth"
            )

            torch.save(
                D.state_dict(),
                CHECKPOINT_DIR /
                f"discriminator_step_{step:05d}.pth"
            )

        if step % 100 == 0:

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()


# ============================================================
# FINAL OUTPUT
# ============================================================

history_df = pd.DataFrame(history)

history_path = WORK_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)

final_sample = save_sample(step)

elapsed_min = (
    time.time() - start_time
) / 60

print()
print("=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print("Steps completed:", step)
print("Best validation step:", best_val_step)
print("Best validation L1:", best_val_l1)
print("Best training-fit step:", best_train_step)
print("Best training L1:", best_train_l1)
print("Training time:", round(elapsed_min, 2), "minutes")
print()
print(
    "Validation checkpoint:",
    CHECKPOINT_DIR / "generator_best.pth"
)
print(
    "Training-fit checkpoint:",
    CHECKPOINT_DIR / "generator_best_trainfit.pth"
)
print("History:", history_path)
print("Final sample:", final_sample)
print("=" * 70)

FINAL PIX2PIX TRAINING
Device: cuda
GPU: Tesla T4
Training samples: 116
Validation samples: 116
Batch size: 8
Maximum steps: 20000
Learning rate: 0.0001
L1 weight: 100.0
Validation every: 250
Samples every: 250
Early stopping: DISABLED
Data sanity check PASSED.
[VALIDATION] Step 00001 | Train L1: 0.147715 | Val L1: 0.150727
[BEST VAL] Step 1 | L1 0.150727
[BEST TRAIN] Step 1 | L1 0.147715
Step 00025/20000 | G 15.3283 | D 0.3509 | L1 0.1365 | Time 0.15 min
Step 00050/20000 | G 16.1523 | D 0.2598 | L1 0.1275 | Time 0.23 min
Step 00075/20000 | G 19.4011 | D 0.2712 | L1 0.1639 | Time 0.32 min
Step 00100/20000 | G 17.8005 | D 0.2741 | L1 0.1493 | Time 0.41 min
Step 00125/20000 | G 17.4963 | D 0.1940 | L1 0.1345 | Time 0.51 min
Step 00150/20000 | G 18.1646 | D 0.1991 | L1 0.1400 | Time 0.60 min
Step 00175/20000 | G 17.6336 | D 0.3400 | L1 0.1340 | Time 0.69 min
Step 00200/20000 | G 17.1144 | D 0.1879 | L1 0.1323 | Time 0.78 min
Step 00225/20000 | G 16.7806 | D 0.1918 | L1 0.1305 | Time 0.87 

In [ ]:
# ============================================================
# CELL 12 — FINAL TRAINING-SET TEST
# ============================================================

BEST_PATH = CHECKPOINT_DIR / "generator_best_trainfit.pth"

if not BEST_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {BEST_PATH}")

checkpoint = torch.load(BEST_PATH, map_location=DEVICE)

if isinstance(checkpoint, dict) and "generator_state_dict" in checkpoint:
    G.load_state_dict(checkpoint["generator_state_dict"])
    print("Checkpoint format: generator_state_dict")
    print("Best step:", checkpoint.get("step", "unknown"))
    print("Best training L1:", checkpoint.get("train_l1", "unknown"))
else:
    G.load_state_dict(checkpoint)
    print("Checkpoint format: raw state_dict")

G = G.to(DEVICE)
G.eval()

test_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_batch = next(iter(test_loader))

thermal = test_batch["thermal"].to(
    DEVICE,
    non_blocking=torch.cuda.is_available()
)

real_rgb = test_batch["rgb"].to(
    DEVICE,
    non_blocking=torch.cuda.is_available()
)

with torch.inference_mode():
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        generated = G(thermal)

print()
print("Thermal shape  :", thermal.shape)
print("Generated shape:", generated.shape)
print("Real RGB shape :", real_rgb.shape)

assert thermal.shape[1:] == (1, IMAGE_SIZE, IMAGE_SIZE)
assert generated.shape[1:] == (3, IMAGE_SIZE, IMAGE_SIZE)
assert real_rgb.shape == generated.shape

n = min(8, thermal.size(0))

thermal_rgb = thermal[:n].repeat(1, 3, 1, 1)

comparison = torch.cat(
    [
        denorm(thermal_rgb),
        denorm(generated[:n]),
        denorm(real_rgb[:n])
    ],
    dim=3
).cpu()

FINAL_SAMPLE = WORK_DIR / "final_train_comparison.png"

save_image(comparison, FINAL_SAMPLE)

print()
print("=" * 70)
print("FINAL CHECK PASSED")
print("=" * 70)
print("Checkpoint:", BEST_PATH)
print("Comparison:", FINAL_SAMPLE)
print("Generated RGB shape:", generated.shape)
print("=" * 70)

## 7. Save inference metadata

The metadata file is uploaded alongside the checkpoint so the Aero-Topo application knows that this is a 1-channel → 3-channel Pix2Pix generator and can reproduce the expected preprocessing.


In [ ]:
# ============================================================
# CELL 13 — CREATE MODEL CONFIG
# ============================================================

model_config = {
    "model_name": "Aero-Topo FLAME 3 Pix2Pix cGAN",
    "architecture": "Pix2Pix U-Net Generator",
    "dataset": "FLAME 3",
    "input": {
        "type": "thermal",
        "channels": 1,
        "normalization": "robust 1st-99th percentile per image",
        "range": [-1, 1],
        "resolution": [IMAGE_SIZE, IMAGE_SIZE]
    },
    "output": {
        "type": "RGB",
        "channels": 3,
        "range": [-1, 1]
    },
    "training": {
        "loss": "BCEWithLogits adversarial + 100 * L1",
        "learning_rate": LEARNING_RATE,
        "beta1": BETA1,
        "beta2": BETA2,
        "batch_size": BATCH_SIZE,
        "max_steps": MAX_STEPS,
        "train_pairs": len(train_df),
        "validation_pairs": len(val_df),
        "augmentation": False
    },
    "checkpoint": {
        "validation_best": "generator_best.pth",
        "training_best": "generator_best_trainfit.pth"
    },
    "best_validation_l1": (
        float(best_val_l1)
        if best_val_l1 != float("inf")
        else None
    ),
    "best_training_l1": (
        float(best_train_l1)
        if best_train_l1 != float("inf")
        else None
    )
}

CONFIG_PATH = WORK_DIR / "config.json"

with open(CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(model_config, f, indent=2)

print("Config saved to:", CONFIG_PATH)
print(json.dumps(model_config, indent=2))

In [ ]:
# ============================================================
# CELL TESTING — RANDOM THERMAL → RGB TEST
# ============================================================

import random
import numpy as np
import torch
from PIL import Image

G.eval()

idx = random.randrange(len(train_dataset))
sample = train_dataset[idx]

thermal = sample["thermal"].unsqueeze(0).to(DEVICE)

with torch.inference_mode():
    with torch.cuda.amp.autocast(enabled=USE_AMP):
        generated = G(thermal)

rgb = denorm(generated[0]).cpu()

rgb = (
    rgb.permute(1, 2, 0).numpy() * 255
).clip(0, 255).astype(np.uint8)

output_path = WORK_DIR / f"random_generated_rgb_{sample['id']}.png"

Image.fromarray(rgb, mode="RGB").save(output_path)

print("=" * 70)
print("RANDOM THERMAL → RGB TEST")
print("=" * 70)
print("Sample ID :", sample["id"])
print("Input     : Thermal IR")
print("Output    : Generated RGB")
print("Shape     :", rgb.shape)
print("Dtype     :", rgb.dtype)
print("Saved to  :", output_path)
print("=" * 70)

## 8. Optional local model inspection

Before uploading, confirm that the checkpoint is a real trained PyTorch state dictionary and that its size is reasonable.


In [ ]:
# ============================================================
# CELL 14 — CHECKPOINT INSPECTION
# ============================================================

size_mb = BEST_PATH.stat().st_size / (1024 ** 2)

print("Checkpoint:", BEST_PATH)
print("Size:", round(size_mb, 2), "MB")
print("Exists:", BEST_PATH.exists())

checkpoint = torch.load(BEST_PATH, map_location="cpu")

print("Checkpoint keys:", list(checkpoint.keys())[:20])

if "generator_state_dict" not in checkpoint:
    raise RuntimeError(
        "Checkpoint does not contain generator_state_dict."
    )

print("Generator parameter tensors:", len(checkpoint["generator_state_dict"]))
print("Checkpoint looks valid.")


# 9. Hugging Face Upload

## Before running this cell

Create a Hugging Face **Model Repository** and set:

```text
HF_REPO_ID = "your-username/aero-topo-flame3-cgan"
```

Then provide the token.

For a public notebook, **do not paste your token into a cell that will be committed/shared**. Kaggle Secrets are preferred.

This cell uploads:

- `generator_best.pth`
- `config.json`
- `final_validation_comparison.png`
- `training_history.csv`

The model repository can then be used by the Aero-Topo application's model downloader.


In [ ]:
# ============================================================
# CELL 15 — PUSH BEST MODEL TO HUGGING FACE
# ============================================================

from pathlib import Path
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient


# ------------------------------------------------------------
# CONFIGURATION
# ------------------------------------------------------------

HF_REPO_ID = "rohithkumarl/FLAME3-DIFFUSION"

BEST_PATH = CHECKPOINT_DIR / "generator_best.pth"
CONFIG_PATH = WORK_DIR / "config.json"
FINAL_SAMPLE = WORK_DIR / "final_validation_comparison.png"
HISTORY_PATH = WORK_DIR / "training_history.csv"


# ------------------------------------------------------------
# LOAD HUGGING FACE TOKEN FROM KAGGLE SECRET
# ------------------------------------------------------------

secrets = UserSecretsClient()

try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
except Exception as e:
    raise RuntimeError(
        "Could not read Kaggle Secret 'HF_TOKEN'.\n"
        "Create a Kaggle secret named exactly: HF_TOKEN"
    ) from e


# ------------------------------------------------------------
# VALIDATION
# ------------------------------------------------------------

if not HF_TOKEN or not HF_TOKEN.strip():
    raise ValueError(
        "Kaggle Secret 'HF_TOKEN' is empty."
    )

if not BEST_PATH.exists():
    raise FileNotFoundError(
        f"Best checkpoint not found:\n{BEST_PATH}"
    )

print("=" * 70)
print("HUGGING FACE UPLOAD")
print("=" * 70)

print("Repository:", HF_REPO_ID)
print("Checkpoint:", BEST_PATH)
print("Checkpoint size:",
      round(BEST_PATH.stat().st_size / (1024 ** 2), 2),
      "MB")


# ------------------------------------------------------------
# CREATE / VERIFY MODEL REPOSITORY
# ------------------------------------------------------------

api = HfApi(
    token=HF_TOKEN
)

api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type="model",
    exist_ok=True
)

print()
print("Hugging Face repository is ready.")


# ------------------------------------------------------------
# FILES TO UPLOAD
# ------------------------------------------------------------

files_to_upload = [
    (
        BEST_PATH,
        "generator_best.pth"
    ),
    (
        CONFIG_PATH,
        "config.json"
    ),
    (
        FINAL_SAMPLE,
        "final_validation_comparison.png"
    ),
    (
        HISTORY_PATH,
        "training_history.csv"
    )
]


# ------------------------------------------------------------
# UPLOAD
# ------------------------------------------------------------

for local_path, repo_path in files_to_upload:

    local_path = Path(local_path)

    if not local_path.exists():

        print(
            f"SKIP — file not found: {local_path}"
        )

        continue

    print(
        f"Uploading {local_path.name} "
        f"→ {repo_path}"
    )

    api.upload_file(
        path_or_fileobj=str(local_path),
        path_in_repo=repo_path,
        repo_id=HF_REPO_ID,
        repo_type="model"
    )

    print(
        f"Uploaded: {repo_path}"
    )


# ------------------------------------------------------------
# DONE
# ------------------------------------------------------------

print()
print("=" * 70)
print("UPLOAD COMPLETE")
print("=" * 70)

print(
    "Repository:",
    HF_REPO_ID
)

print(
    "Main model:",
    "generator_best.pth"
)

print("=" * 70)